In [ ]:
# Q5B: AdaBoost, Gradient Boosting, XGBoost, and hard/soft voting on Iris.
# Source: https://www.kaggle.com/datasets/uciml/iris
# Install xgboost in a new environment if needed: pip install xgboost
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier


In [ ]:
# Load the existing Iris CSV and encode the three species as integers.
iris = pd.read_csv("Iris.csv")
X = iris.drop(columns=["Id", "Species"])
encoder = LabelEncoder()
y = encoder.fit_transform(iris["Species"])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)
print("Training samples:", len(X_train), "Test samples:", len(X_test))
print("Species:", list(encoder.classes_))


In [ ]:
# Build the three requested boosting models.
models = {
    "AdaBoost": AdaBoostClassifier(n_estimators=60, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=80, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=80, max_depth=3, learning_rate=0.1,
                              objective="multi:softprob", num_class=3,
                              eval_metric="mlogloss", n_jobs=1, random_state=42),
}


In [ ]:
# Compare each model using the same held-out test set.
results = []
fitted_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model
    predicted = model.predict(X_test)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predicted),
        "Macro precision": precision_score(y_test, predicted, average="macro", zero_division=0),
        "Macro recall": recall_score(y_test, predicted, average="macro", zero_division=0),
        "Macro F1": f1_score(y_test, predicted, average="macro", zero_division=0),
    })


In [ ]:
# Hard voting uses majority class labels; soft voting averages probabilities.
for voting in ["hard", "soft"]:
    ensemble = VotingClassifier(estimators=list(models.items()), voting=voting)
    ensemble.fit(X_train, y_train)
    predicted = ensemble.predict(X_test)
    fitted_models[f"{voting.title()} voting"] = ensemble
    results.append({
        "Model": f"{voting.title()} voting",
        "Accuracy": accuracy_score(y_test, predicted),
        "Macro precision": precision_score(y_test, predicted, average="macro", zero_division=0),
        "Macro recall": recall_score(y_test, predicted, average="macro", zero_division=0),
        "Macro F1": f1_score(y_test, predicted, average="macro", zero_division=0),
    })
comparison = pd.DataFrame(results).set_index("Model")
print(comparison.round(3).to_string())
comparison["Macro F1"].sort_values().plot.barh(figsize=(7, 4), xlim=(0, 1),
                                                 title="Iris model comparison")
plt.tight_layout()
plt.show()


In [ ]:
# Inspect predictions from the top-scoring model.
best_name = comparison["Macro F1"].idxmax()
best_predictions = fitted_models[best_name].predict(X_test)
print("Best model:", best_name)
ConfusionMatrixDisplay.from_predictions(y_test, best_predictions,
                                        display_labels=encoder.classes_, cmap="Blues")
plt.title(best_name + " confusion matrix")
plt.tight_layout()
plt.show()
